# Notebook 02: Microbiota-Transcriptome PC Associations

Core association testing between microbiome PCs/residuals and transcriptome PCs/residuals.

** Four comparisons from the Figure 2 framework: **
- **Part 2:** microbiome principal components (mPCs) vs host gene expression principal components (gPCs)
- **Part 3:** microbiome corrected for leading mPCs (aka microbiome residuals, mResid) vs gene expression corrected for leading gPCs (aka transcriptome residuals, gResid)
- **Part 4:** mResid vs gPCs
- **Part 5:** mPCs vs gResid

Each part tests 3 intestinal locations (IL, TR, RE) and 6 immune cell types (CD4-PLA) separately,
with Hochberg multiple adjustment correction. Parts 3-5 also include genomic inflation correction due to their high dimensionality.

## Bootstrap: Environment Setup

In [ ]:
# --- Bootstrap: replicate 00_setup.ipynb environment ---
BASE_DIR <- normalizePath(file.path(getwd(), '..'))
DATA_DIR    <- file.path(BASE_DIR, 'Data')
CODE_DIR    <- file.path(BASE_DIR, 'code', 'CEDAR-master')
OUTPUT_DIR  <- file.path(BASE_DIR, 'notebooks', 'output')
PLOTS_DATA  <- file.path(BASE_DIR, 'code', 'data_for_plots')
TRANSCR_DIR       <- file.path(DATA_DIR, 'CEDAR1_transcriptome')
EXPR_PC_CORR_DIR  <- file.path(TRANSCR_DIR, 'Expr_data_Corr_for_PCs')
TRANSCR_PCS_DIR   <- file.path(TRANSCR_DIR, 'Momo-Dmitrieva2018')
MODULE_EIGEN_DIR  <- file.path(TRANSCR_DIR, 'Module_eigengenes_expr')

dir.create(OUTPUT_DIR, showWarnings = FALSE, recursive = TRUE)

suppressPackageStartupMessages({
    library(data.table)
    library(ade4)
    library(vegan)
    library(chemometrics)
    library(ggplot2)
    library(igraph)
    library(gplots)
    library(ape)
    library(stringr)
    library(jsonlite)
    library(dplyr)
    library(lattice)
    library(RhpcBLASctl)
    library(data.table)
    library(parallel)
    library(doParallel)
    library(foreach)
})
for (pkg in c('WGCNA', 'Rtsne', 'DirichletMultinomial', 'WebGestaltR', 'RCurl')) {
    if (requireNamespace(pkg, quietly = TRUE)) {
        suppressPackageStartupMessages(library(pkg, character.only = TRUE))
    }
}

R_DIR <- file.path(BASE_DIR, 'notebooks', 'R')
source(file.path(R_DIR, 'functions_general.R'))
source(file.path(R_DIR, 'functions_figures.R'))
source(file.path(R_DIR, 'functions_association.R'))
source(file.path(R_DIR, 'functions_transcriptome.R'))

LOCATIONS   <- c('IL', 'TR', 'RE')
CELL_TYPES  <- c('CD4', 'CD8', 'CD14', 'CD15', 'CD19', 'PLA')
ALL_TYPES   <- c(LOCATIONS, CELL_TYPES)
N_TRANSCR_PCS_LOCATION <- c(IL = 59, TR = 50, RE = 53)
N_TRANSCR_PCS_CELL_TYPE <- c(CD4 = 38, CD8 = 35, CD14 = 36,
                          CD15 = 27, CD19 = 40, PLA = 23)
N_TRANSCR_PCS <- c(N_TRANSCR_PCS_LOCATION, N_TRANSCR_PCS_CELL_TYPE)
N_MICROB_PCS_LOCATION <- 15
N_MICROB_PCS_CELL_TYPE <- 4

N_SIDAK_PERM = 1000

In [ ]:
# --- Setup Parallel Backend ---
num_cores <- 30

## Load and Prepare Input Data

Two types of microbiome input:
- **microbiome.15PCs**: first 15 columns of `taxa_abund_clr_corrected_in_20PC_coord.tsv` (samples × 15 PCs)
- **taxa.abund.corr15PCs_t**: transposed `taxa_abund_clr_corrected_15PCs.tsv` (samples × 145 taxa residuals)

Transcriptome files are read on-the-fly by `corr.trPCs.vs.mPCs()` from:
- **TRANSCR_PCS_DIR** (`Momo-Dmitrieva2018/`): transcriptome PCs (Parts 1, 3)
- **EXPR_PC_CORR_DIR** (`Expr_data_Corr_for_PCs/`): expression residuals (Parts 2, 4)

In [ ]:
# Output subdirectory for this notebook
NB02_OUT <- file.path(OUTPUT_DIR, 'nb02')
dir.create(NB02_OUT, showWarnings = FALSE, recursive = TRUE)

In [ ]:
# --- Microbiome PCs (for Parts 2 and 5) ---
# Read the 20PC coordinate file, keep first 15 PCs
microbiome.20PCs <- read.table.smart(file.path(DATA_DIR, 'taxa_abund_clr_corrected_in_20PC_coord.tsv'))
microbiome.15PCs <- microbiome.20PCs[, 1:15]
cat('microbiome.15PCs:', nrow(microbiome.15PCs), 'samples x', ncol(microbiome.15PCs), 'PCs\n')
cat('Column names:', paste(colnames(microbiome.15PCs), collapse = ', '), '\n')

# --- Microbiome residuals (for Parts 3 and 4) ---
# Read taxa corrected for 15 PCs, transpose to samples x taxa
taxa.abund.corr15PCs <- read.table.smart(file.path(DATA_DIR, 'taxa_abund_clr_corrected_15PCs.tsv'))
taxa.abund.corr15PCs_t <- as.data.frame(t(taxa.abund.corr15PCs))
cat('taxa.abund.corr15PCs_t:', nrow(taxa.abund.corr15PCs_t), 'samples x', ncol(taxa.abund.corr15PCs_t), 'taxa\n')

microbiome.15PCs: 315 samples x 15 PCs
Column names: RS1, RS2, RS3, RS4, RS5, RS6, RS7, RS8, RS9, RS10, RS11, RS12, RS13, RS14, RS15 
taxa.abund.corr15PCs_t: 315 samples x 145 taxa

Pre-computed significant results: 11 associations


In [ ]:
# --- Scree plot: microbiome PCA variance explained (Suppl. Fig. 7) ---
# Re-run PCA on the CLR-corrected abundances to get eigenvalues.
# taxa_abund_clr_corrected.tsv: samples x taxa
clr.abund <- read.table.smart(file.path(DATA_DIR, 'taxa_abund_clr_corrected.tsv'))

pca.mb <- prcomp(clr.abund, scale. = FALSE)  # CLR data already on comparable scale
var.expl <- pca.mb$sdev^2 / sum(pca.mb$sdev^2) * 100
cumvar <- cumsum(var.expl)

n_show <- 20 # show first N PCs

pdf(file.path(NB02_OUT, 'scree_microbiome_PCs.pdf'), width = 10, height = 5)
par(mfrow = c(1, 2), mar = c(5, 4, 3, 1))

# Left: per-PC variance
barplot(var.expl[1:n_show],
        names.arg = paste0('mPC', 1:n_show),
        las = 2, col = 'steelblue',
        ylab = 'Variance explained (%)',
        #main = 'Microbiome PCs: scree plot',
        cex.names = 0.75)

# Right: cumulative variance with PC thresholds marked
plot(1:n_show, cumvar[1:n_show], type = 'b', pch = 16,
xlab = 'Number of PCs', ylab = 'Cumulative variance (%)',
#main = 'Microbiome PCs: cumulative variance',
ylim = c(0, 100))
abline(h   = c(44.9), lty = 2, col = 'gray60')
abline(v   = c(15),      lty = 2, col = 'firebrick')

dev.off()

cat('Variance explained by first 15 PCs:', round(cumvar[15], 1), '%\n')
cat('Variance explained by first  4 PCs:', round(cumvar[4],  1), '%\n')

pdf 
  2

Variance explained by first 15 PCs: 44.9 %
Variance explained by first  4 PCs: 24.5 %


In [ ]:
# --- Preparation: Read once ---

# gPCs
transcriptome_list <- list()
for(tn in ALL_TYPES){
    fname <- paste0(word(tn, 1, sep = '\\.'), '_PCs_after_correction_v2.txt')
    transcriptome_list[[tn]] <- read.table(file.path(TRANSCR_PCS_DIR, fname), row.names = 1, header = T)
}

In [ ]:
# gResid
# 3 min
transcriptome_list_residuals <- list()
for(tn in ALL_TYPES){
    fname <- paste0(word(tn, 1, sep = '\\.'), '_GE_Corrected4_Covars_PCs.txt')
    transcriptome_list_residuals[[tn]] <- read.table(file.path(EXPR_PC_CORR_DIR, fname), row.names = 1, header = T)
}

In [9]:
# 1. Setup
target_names <- c(LOCATIONS, CELL_TYPES)
n_pcs_combined <- c(N_TRANSCR_PCS_LOCATION, N_TRANSCR_PCS_CELL_TYPE)

## Part 1: Microbiome 15 PCs vs Gene Expression PCs

Test association between 15 microbiome PCs (from PCA of CLR-corrected
abundances) and transcriptome PCs (from PCA of covariate-corrected gene expression).

- Method: `lm(gPC ~ mPC)` via `anova()` for p-values
- Outlier removal: **only transcriptome** side (`remove.outliers = c(F, T)`), IQR × 5
- No inflation correction (not needed for PC-vs-PC with modest test count)
- Number of mPCs: 15 (for all types)
- Number of gPCs: 59 (IL), 50 (TR), 53 (RE), 38 (CD4), 35 (CD8), 36 (CD14), 27 (CD15), 40 (CD19), 23 (PLA)

In [ ]:
# --- Part 1a (3 Tissues) ---
# Prepare
prepared_part1a <- prepare_data_universal(
    microb_df = microbiome.15PCs, 
    transcr_list = transcriptome_list[c(LOCATIONS)],
    n_pcs = c(N_TRANSCR_PCS_LOCATION),
    exclude_mode = 'last_columns',
    outlier_modes = c(FALSE, TRUE),
    iqr_mult = 5
)

In [ ]:
# Calculate Real P-values
pval_real_part1a <- run_universal_association(
	prepared_part1a,
	m_prefix="mPC",
	t_prefix="gPC",
	inflation_correct = FALSE,
	QQplot.file = file.path(NB02_OUT, 'QQ_Part1a_mPCs_gPCs_3locs.pdf'),
    main = 'Taxonomic PCs vs gene expression PCs (3 locations)'
)

In [ ]:
# (not needed in Part 1a) Extract the lambda that was calculated during the association step

In [ ]:
# Run the Neff Pipeline
res_part1a <- run_neff_pipeline(
    pval_observed = pval_real_part1a, 
    prepared_data = prepared_part1a, 
    microb_base = microbiome.15PCs,
    n_perm = N_SIDAK_PERM,
    n_cores = num_cores,
    fixed_lambda = NULL # no inflation correction
)

Starting Neff estimation with 1000 permutations on 30 cores...
Neff coefficient of variation:  0.0134 
Neff Estimation Complete (2.5s)
Total Tests: 2430 
Effective Independent Tests (Neff): 2422.34 
Reduction in penalty: 0.3 %

p Sidak <= 0.15: 0 
named numeric(0)

p Hochberg <= 0.15: 0 
named numeric(0)


In [ ]:
# 
write.pvalue.vectors.to.file(
    output_file = file.path(NB02_OUT, 'pvalues_Part1a_mPCs_gPCs_3locs.tsv'),
    Original = pval_real_part1a,
    Hochberg = res_part1a$pval_hochberg,
    Sidak = res_part1a$pval_sidak,
    signif_ = 4
)

In [ ]:
prepared_part1b <- prepare_data_universal(
    microb_df = microbiome.15PCs, 
    transcr_list = transcriptome_list[c(CELL_TYPES)],
    n_pcs = c(N_TRANSCR_PCS_CELL_TYPE),
    exclude_mode = 'last_columns',
    outlier_modes = c(FALSE, TRUE),
    iqr_mult = 5   
)

In [ ]:
pval_real_part1b <- run_universal_association(
	prepared_part1b,
	m_prefix="mPC",
	t_prefix="gPC",
	inflation_correct = FALSE,
	QQplot.file = file.path(NB02_OUT, 'QQ_Part1b_mPCs_gPCs_6cells.pdf'),
    main = 'Taxonomic PCs vs gene expression PCs (6 immune cell types)'
)

In [ ]:
# (not needed in Part 1b) Extract the lambda that was calculated during the association step

In [ ]:
res_part1b <- run_neff_pipeline(
    pval_observed = pval_real_part1b, 
    prepared_data = prepared_part1b, 
    microb_base = microbiome.15PCs, # Shuffling the 15 microbiome PCs
    n_perm = N_SIDAK_PERM,
    n_cores = num_cores
)

Starting Neff estimation with 1000 permutations on 30 cores...
Neff coefficient of variation:  0.0218 
Neff Estimation Complete (3.2s)
Total Tests: 2985 
Effective Independent Tests (Neff): 2718.81 
Reduction in penalty: 8.9 %

p Sidak <= 0.15: 1 
CD8.mPC11.gPC5 
    0.03502928 

p Hochberg <= 0.15: 1 
CD8.mPC11.gPC5 
    0.03914843 


In [ ]:
# 
write.pvalue.vectors.to.file(
    output_file = file.path(NB02_OUT, 'pvalues_Part1b_mPCs_gPCs_6cells.tsv'),
    Original = pval_real_part1b,
    Hochberg = res_part1b$pval_hochberg,
    Sidak = res_part1b$pval_sidak,
    signif_ = 4
)

## Part 2: Microbiome Residuals vs Gene Expression Residuals

Test association between individual taxa abundances (after adjusting for
15 microbiome PCs) and individual gene expression residuals (after removing covariates + PCs).

- Microbiome: 145 taxa residuals from `taxa_abund_clr_corrected_15PCs.tsv`
- Transcriptome: PC-corrected expression from `Expr_data_Corr_for_PCs/`
- Method: `lm(gene ~ taxon)` via `anova()` for p-values
- Outlier removal: **only microbiome** side (`remove.outliers = c(T, F)`), IQR × 5
- Inflation correction: yes (both raw and corrected p-values computed)

In [ ]:
prepared_part2a <- prepare_data_universal(
    microb_df = taxa.abund.corr15PCs_t,
    transcr_list = transcriptome_list_residuals[c(LOCATIONS)],
    n_pcs = NULL,
    exclude_mode = 'first_1_column',
    outlier_modes = c(TRUE, FALSE),
    iqr_mult = 5
)

In [ ]:
# 3 min
pval_real_part2a <- run_universal_association(
	prepared_part2a,
	m_prefix="",
	t_prefix="",
	inflation_correct = TRUE,
	QQplot.file = file.path(NB02_OUT, 'QQ_Part2a_mResid_gResid_3locs.pdf'),
    main = 'Taxonomic residuals vs gene expression residuals (3 locations)'
)

In [ ]:
# Extract the lambda that was calculated during the association step
real_lambda_part2a <- attr(pval_real_part2a, "lambda")
real_lambda_part2a

[1] 0.984937

In [ ]:
# 36 min
res_part2a <- run_neff_pipeline(
    pval_observed = pval_real_part2a, 
    prepared_data = prepared_part2a, 
    microb_base = taxa.abund.corr15PCs_t,
    n_perm = N_SIDAK_PERM,
    n_cores = num_cores,
    ##inflation_correct = TRUE,
    fixed_lambda = real_lambda_part2a # Use the same correction factor
)

Starting Neff estimation with 1000 permutations on 30 cores...
Neff coefficient of variation:  7e-04 
Neff Estimation Complete (2153.4s)
Total Tests: 6572415 
Effective Independent Tests (Neff): 96308342.35 
Reduction in penalty: -1365.3 %

p Sidak <= 0.15: 0 
named numeric(0)

p Hochberg <= 0.15: 0 
named numeric(0)


In [ ]:
# 40 sec
write.pvalue.vectors.to.file(
    output_file = file.path(NB02_OUT, 'pvalues_Part2a_mResid_gResid_3locs.tsv'),
    Original = pval_real_part2a,
    Hochberg = res_part2a$pval_hochberg,
    Sidak = res_part2a$pval_sidak,
    ###Infl.corrected = NULL,
    ###Infl.corrected.Hochberg = NULL,
    signif_ = 4
)

In [ ]:
prepared_part2b <- prepare_data_universal(
    microb_df = taxa.abund.corr15PCs_t,
    transcr_list = transcriptome_list_residuals[c(CELL_TYPES)],
    n_pcs = NULL,
    exclude_mode = 'first_1_column',
    outlier_modes = c(TRUE, FALSE),
    iqr_mult = 5
)

In [ ]:
# 6 min
pval_real_part2b <- run_universal_association(
	prepared_part2b,
	m_prefix="",
	t_prefix="",
	inflation_correct = TRUE,
	QQplot.file = file.path(NB02_OUT, 'QQ_Part2b_mResid_gResid_6cells.pdf'),
    main = 'Taxonomic residuals vs gene expression residuals (6 immune cell types)'
)

In [ ]:
# Extract the lambda that was calculated during the association step
real_lambda_part2b <- attr(pval_real_part2b, "lambda")
real_lambda_part2b

[1] 0.9975658

In [ ]:
# 59 min
res_part2b <- run_neff_pipeline(
    pval_observed = pval_real_part2b, 
    prepared_data = prepared_part2b, 
    microb_base = taxa.abund.corr15PCs_t,
    n_perm = N_SIDAK_PERM,
    n_cores = num_cores,
    ##inflation_correct = TRUE,
    fixed_lambda = real_lambda_part2b # Use the same correction factor
)

Starting Neff estimation with 1000 permutations on 30 cores...
Neff coefficient of variation:  0.0534 
Neff Estimation Complete (3563s)
Total Tests: 10194660 
Effective Independent Tests (Neff): 256787187.92 
Reduction in penalty: -2418.8 %

p Sidak <= 0.15: 0 
named numeric(0)

p Hochberg <= 0.15: 2 
 CD8.k__Bacteria;p__Firmicutes;c__Clostridia;o__Clostridiales;f__Lachnospiraceae;g__Lachnospiraceae.UCG.008.X2070131 
                                                                                                          0.03522384 
CD19.k__Bacteria;p__Firmicutes;c__Clostridia;o__Clostridiales;f__Ruminococcaceae;g__Ruminococcaceae.UCG.003.X2650022 
                                                                                                          0.04040064 


## Part 3: Microbiome Residuals vs Gene Expression PCs

Test association between individual taxa (genera) residuals and transcriptome PCs.

- Microbiome: 145 taxa residuals (same as Part 2)
- Transcriptome: PCs from `Momo-Dmitrieva2018/` (same files as Part 1)
- Method: `lm(gPC ~ taxon)` via `anova()`
- Outlier removal: **both** sides (`remove.outliers = c(T, T)`), IQR × 5
- Inflation correction: yes

In [ ]:
prepared_part3a <- prepare_data_universal(
    microb_df = taxa.abund.corr15PCs_t,
    transcr_list = transcriptome_list[c(LOCATIONS)],
    n_pcs = c(N_TRANSCR_PCS_LOCATION),
    exclude_mode = 'last_columns',
    outlier_modes = c(TRUE, TRUE),
    iqr_mult = 5
)

In [ ]:
# 1 sec
pval_real_part3a <- run_universal_association(
	prepared_part3a, 
	m_prefix="", 
	t_prefix="gPC", 
	inflation_correct = TRUE,	
	QQplot.file = file.path(NB02_OUT, 'QQ_Part3a_mResid_gPCs_3locs.pdf'),
    main = 'Taxonomic residuals vs gPCs (3 locations)'
)

In [ ]:
# Extract the lambda that was calculated during the association step
real_lambda_part3a <- attr(pval_real_part3a, "lambda")
real_lambda_part3a

[1] 1.013059

In [ ]:
# 1 min
res_part4a <- run_neff_pipeline(
    pval_observed = pval_real_part3a, 
    prepared_data = prepared_part3a, 
    microb_base = taxa.abund.corr15PCs_t,
    n_perm = 10000, #N_SIDAK_PERM,
    n_cores = num_cores,
    #inflation_correct = TRUE,
    fixed_lambda = real_lambda_part2a # Use the same correction factor
)

Starting Neff estimation with 10000 permutations on 30 cores...
Neff coefficient of variation:  0.0163 
Neff Estimation Complete (81.1s)
Total Tests: 23490 
Effective Independent Tests (Neff): 28416.25 
Reduction in penalty: -21 %

p Sidak <= 0.15: 1 
IL.k__Bacteria;p__Firmicutes;c__Clostridia;o__Clostridiales;f__Lachnospiraceae;g__Moryella.gPC7 
                                                                                      0.1050374 

p Hochberg <= 0.15: 1 
IL.k__Bacteria;p__Firmicutes;c__Clostridia;o__Clostridiales;f__Lachnospiraceae;g__Moryella.gPC7 
                                                                                     0.09173483 


In [ ]:
write.pvalue.vectors.to.file(
    output_file = file.path(NB02_OUT, 'pvalues_Part3a_mResid_gPCs_3locs.tsv'),
    Original = pval_real_part3a,
    Hochberg = res_part3a$pval_hochberg,
    Sidak = res_part3a$pval_sidak,
    Infl.corrected = NULL,
    Infl.corrected.Hochberg = NULL,
    signif_ = 4
)

In [ ]:
prepared_part3b <- prepare_data_universal(
    microb_df = taxa.abund.corr15PCs_t,
    transcr_list = transcriptome_list[c(CELL_TYPES)],    
    n_pcs = c(N_TRANSCR_PCS_CELL_TYPE),
    exclude_mode = 'last_columns',
    outlier_modes = c(TRUE, TRUE)
)

In [ ]:
# 1 sec
pval_real_part3b <- run_universal_association(
	prepared_part3b, 
	m_prefix="", 
	t_prefix="gPC", 
	inflation_correct = TRUE,
	QQplot.file = file.path(NB02_OUT, 'QQ_Part3b_mResid_gPCs_6cells.pdf'),
    main = 'Taxonomic residuals vs gPCs (6 immune cell types)'
)

In [ ]:
# Extract the lambda that was calculated during the association step
real_lambda_part3b <- attr(pval_real_part3b, "lambda")
real_lambda_part3b

[1] 1.034179

In [ ]:
# 13 sec
res_part4b <- run_neff_pipeline(
    pval_observed = pval_real_part3b, 
    prepared_data = prepared_part3b, 
    microb_base = taxa.abund.corr15PCs_t,
    n_perm = N_SIDAK_PERM,
    n_cores = num_cores,
    fixed_lambda = real_lambda_part3b # Use the same correction factor
)

Starting Neff estimation with 1000 permutations on 30 cores...
Neff coefficient of variation:  0.0272 
Neff Estimation Complete (13.1s)
Total Tests: 28855 
Effective Independent Tests (Neff): 29740.05 
Reduction in penalty: -3.1 %

p Sidak <= 0.15: 0 
named numeric(0)

p Hochberg <= 0.15: 0 
named numeric(0)


In [ ]:
write.pvalue.vectors.to.file(
    output_file = file.path(NB02_OUT, 'pvalues_Part3b_mResid_gPCs_6cells.tsv'),
    Original = pval_real_part4b,
    Hochberg = res_part4b$pval_hochberg,
    Sidak = res_part4b$pval_sidak,
    Infl.corrected = NULL,
    Infl.corrected.Hochberg = NULL,
    signif_ = 4
)

## Part 4: Microbiome 15 PCs vs Gene Expression Residuals

Test association between 15 microbiome PCs and individual gene
expression residuals (PC-corrected).

- Microbiome: 15 PCs (same as Part 1)
- Transcriptome: PC-corrected expression from `Expr_data_Corr_for_PCs/` (same files as Part 2)
- Method: `lm(gene ~ mPC)` via `anova()`
- Outlier removal: **neither** side (`remove.outliers = c(F, F)`), IQR × 5
- Inflation correction: yes

In [ ]:
prepared_part4a <- prepare_data_universal(
    microb_df = microbiome.15PCs, 
    transcr_list = transcriptome_list_residuals[c(LOCATIONS)],
    exclude_mode = 'first_1_column', # Residuals usually have ID as first column
    outlier_modes = c(FALSE, FALSE), # No outlier removal for Part 5
    iqr_mult = 5
)

In [ ]:
# 20 sec
pval_real_part4a <- run_universal_association(
	prepared_part4a,
	m_prefix="mPC",
	t_prefix="",
	inflation_correct = TRUE,	
	QQplot.file = file.path(NB02_OUT, 'QQ_Part4a_mPCs_gResid_3locs.pdf'),
    main = 'Taxonomic PCs vs gene expression residuals (3 locations)'
)

In [ ]:
# Extract the lambda that was calculated during the association step
real_lambda_part4a <- attr(pval_real_part4a, "lambda")
real_lambda_part4a

[1] 0.9687451

In [ ]:
# 4 min
res_part4a <- run_neff_pipeline(
    pval_observed = pval_real_part4a, 
    prepared_data = prepared_part4a, 
    microb_base = microbiome.15PCs, 
    n_perm = N_SIDAK_PERM,
    n_cores = num_cores,
    #inflation_correct = TRUE,
    fixed_lambda = real_lambda_part4a # Use the same correction factor
)

Starting Neff estimation with 1000 permutations on 30 cores...
Neff coefficient of variation:  0.0123 
Neff Estimation Complete (226.7s)
Total Tests: 679905 
Effective Independent Tests (Neff): 1003319.02 
Reduction in penalty: -47.6 %

p Sidak <= 0.15: 1 
RE.mPC5.X2070368 
      0.07685114 

p Hochberg <= 0.15: 1 
RE.mPC5.X2070368 
       0.0541886 


In [ ]:
write.pvalue.vectors.to.file(
    output_file = file.path(NB02_OUT, 'pvalues_Part4a_mPCs_gResid_3locs.tsv'),
    Original = pval_real_part4a,
    Hochberg = res_part4a$pval_hochberg,
    Sidak = res_part4a$pval_sidak,
    Infl.corrected = NULL,
    Infl.corrected.Hochberg = NULL,
    signif_ = 4
)

In [ ]:
prepared_part4b <- prepare_data_universal(
    microb_df = microbiome.15PCs, 
    transcr_list = transcriptome_list_residuals[c(CELL_TYPES)],
    exclude_mode = 'first_1_column',
    outlier_modes = c(FALSE, FALSE), # No outlier removal for Part 5
    iqr_mult = 5
)

In [ ]:
# 30 sec
pval_real_part4b <- run_universal_association(
	prepared_part4b,
	m_prefix="mPC",
	t_prefix="",
	inflation_correct = TRUE,	
	QQplot.file = file.path(NB02_OUT, 'QQ_Part4b_mPCs_gResid_6cells.pdf'),
    main = 'Taxonomic PCs vs gene expression residuals (6 immune cell types)'
)

In [ ]:
# Extract the lambda that was calculated during the association step
real_lambda_part4b <- attr(pval_real_part4b, "lambda")
real_lambda_part4b

[1] 1.01681

In [ ]:
# 6.5 min
res_part4b <- run_neff_pipeline(
    pval_observed = pval_real_part4b, 
    prepared_data = prepared_part4b,
    microb_base = microbiome.15PCs, 
    n_perm = N_SIDAK_PERM,
    n_cores = num_cores,
    #inflation_correct = TRUE,
    fixed_lambda = real_lambda_part4b # Use the same correction factor
)

Starting Neff estimation with 1000 permutations on 30 cores...
Neff coefficient of variation:  0.01 
Neff Estimation Complete (385.9s)
Total Tests: 1054620 
Effective Independent Tests (Neff): 886865.96 
Reduction in penalty: 15.9 %

p Sidak <= 0.15: 0 
named numeric(0)

p Hochberg <= 0.15: 0 
named numeric(0)


In [ ]:
write.pvalue.vectors.to.file(
    output_file = file.path(NB02_OUT, 'pvalues_Part4b_mPCs_gResid_6cells.tsv'),
    Original = pval_real_part4b,
    Hochberg = res_part4b$pval_hochberg,
    Sidak = res_part4b$pval_sidak,
    Infl.corrected = NULL,
    Infl.corrected.Hochberg = NULL,
    signif_ = 4
)

## Add-on: Neff Testing

In [ ]:
run_neff_test <- function(n_tests = 100, n_perms = 1200, scenario = "independent") {
  cat("\nTesting Scenario:", scenario, "(Actual N =", n_tests, ")\n")
  
  if (scenario == "independent") {
    # Nominal p-values: 100 independent draws from Uniform(0,1)
    p_nominal <- runif(n_tests)
    
    # Permutation min p-values: min of 100 independent uniforms per perm
    # (Mathematically, this follows a Beta(1, n_tests) distribution)
    perm_min_p <- apply(matrix(runif(n_tests * n_perms), ncol = n_tests), 1, min)
    
  } else if (scenario == "redundant") {
    # Nominal p-values: 100 identical tests (all same p-value)
    p_single <- runif(1)
    p_nominal <- rep(p_single, n_tests)
    
    # Permutation min p-values: since all tests are identical, min(p) is just the p-value of one test
    perm_min_p <- runif(n_perms)
  }
  
  neff <- estimate_neff_sidak(p_nominal, perm_min_p)
  cat("Estimated Neff:", round(neff, 2), "\n")
  
  # Validation logic
  if (scenario == "independent") {
    # Allow some noise due to random sampling, but should be close to n_tests
    success <- abs(neff - n_tests) < (n_tests * 0.2) 
  } else {
    success <- abs(neff - 1) < 0.5
  }
  
  cat("Test Result:", if(success) "PASSED" else "FAILED", "\n")
}

# --- 3. Execute Tests ---
set.seed(123) # For reproducibility
run_neff_test(n_tests = 500, scenario = "independent")
run_neff_test(n_tests = 500, scenario = "redundant")